In [ ]:
# Modules utilisés
import pandas as pd
import requests

# Options Pandas
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option("display.max_colwidth", None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_columns', 200)

In [ ]:
url = "https://data.ameli.fr/api/explore/v2.1/catalog/datasets/demographie-effectifs-et-les-densites/records"

all_results = []
offset = 0

while True:

    params = {
        "limit": 100,
        "offset": offset,
        "where": "year(annee)=2024 AND classe_age='tout_age' AND libelle_sexe='tout sexe'" 
    }

    response = requests.get(url, params=params)
    data = response.json()
    max_results = data["total_count"]
    donnees_specialites = data["results"]

    all_results.extend(donnees_specialites)

    offset += 100
    if offset >= max_results:
        break

df_medecins = pd.DataFrame(all_results)

In [ ]:
df_medecins.to_csv("../data/raw/toutes_specialites_departement.csv", index=False, encoding="utf-8-sig")

In [ ]:
df_medecins_trie = df_medecins[df_medecins["profession_sante"].isin([
    "Chirurgiens-dentistes (hors spécialistes d'orthopédie dento-faciale - ODF)",
    "Cardiologues",
    "Dermatologues",
    "Ophtalmologues",
    "Neurologues",
    "Hépato-gastro-entérologues",
    "Pneumologues",
    "Gynécologues médicaux et obstétriciens",
    "Médecins généralistes (hors médecins à expertise particulière - MEP)",
    "Psychiatres"
])]

In [ ]:
df_medecins_trie = df_medecins_trie[df_medecins_trie['departement'] != '999']

In [ ]:
df_medecins_trie

In [ ]:
df_specialisations_clean = df_medecins_trie.pivot_table(
    values="densite",
    index="libelle_departement",
    columns="profession_sante"
)

In [ ]:
df_specialisations_clean = df_specialisations_clean.reset_index()

In [ ]:
df_specialisations_clean = df_specialisations_clean.rename(columns={
    "Chirurgiens-dentistes (hors spécialistes d'orthopédie dento-faciale - ODF)" : "Chirurgiens-dentistes",
    "Médecins généralistes (hors médecins à expertise particulière - MEP)": "Médecins généralistes"
})

In [ ]:
df_specialisations_clean.head(5)

In [ ]:
df_specialisations_clean.to_csv("../data/clean/specialisations_medecins_clean.csv", index=False, encoding="utf-8-sig")